<a href="https://colab.research.google.com/github/DariushEB/PANI-CNT-Thermoelectric-composite/blob/main/polynominal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm  # For p-values

In [3]:
data = pd.read_excel('low_features_dataset.xlsx')
data.head(5)

,Material (M),Electrolyte,E_Concentration (M),Is_Binder,Morphology_Encoded,Current_Density (A/g),Specific_Capacitance (Fg-1),DOI (Reference)
0,1,3,0.5,1,2,0.5,375.0,https://doi.org/10.1063/5.0086344
1,1,3,0.5,1,2,1.0,175.0,https://doi.org/10.1063/5.0086344
2,1,3,0.5,1,2,1.5,100.0,https://doi.org/10.1063/5.0086344
3,1,3,0.5,1,2,2.0,70.0,https://doi.org/10.1063/5.0086344
4,1,3,1.0,1,1,1.0,347.0,https://doi.org/10.1016/j.jelechem.2020.114080


In [4]:
data= pd.DataFrame(data)
data=data.drop('DOI (Reference)', axis=1)
data.head(5)

,Material (M),Electrolyte,E_Concentration (M),Is_Binder,Morphology_Encoded,Current_Density (A/g),Specific_Capacitance (Fg-1)
0,1,3,0.5,1,2,0.5,375.0
1,1,3,0.5,1,2,1.0,175.0
2,1,3,0.5,1,2,1.5,100.0
3,1,3,0.5,1,2,2.0,70.0
4,1,3,1.0,1,1,1.0,347.0


In [5]:
data.columns

Index(['Material (M)', 'Electrolyte', 'E_Concentration (M)', 'Is_Binder',
       'Morphology_Encoded', 'Current_Density (A/g)',
       'Specific_Capacitance (Fg-1)'],
      dtype='object')

In [6]:
X=data[['Material (M)', 'Electrolyte', 'E_Concentration (M)', 'Is_Binder',
       'Morphology_Encoded', 'Current_Density (A/g)']]
Y=data[['Specific_Capacitance (Fg-1)']]

In [7]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [8]:
print('X_Train',X_train)

X_Train      Material (M)  Electrolyte  E_Concentration (M)  Is_Binder  \
22              1            2                  1.0          1   
15              1            3                  1.0          1   
65              1            2                  3.5          1   
11              1            3                  1.0          1   
42              1            3                  1.0          1   
..            ...          ...                  ...        ...   
71              1            1                  1.0          1   
106             2            3                  0.5          1   
14              1            3                  1.0          1   
92              2            2                  6.0          1   
102             2            2                  6.0          1   

     Morphology_Encoded  Current_Density (A/g)  
22                    3                  0.005  
15                    1                  3.000  
65                    2                  2.000  
11 

In [9]:
# Apply Polynomial Features
poly = PolynomialFeatures (degree = 2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

In [10]:
# Train a model (e.g., Linear Regression)
model = LinearRegression()
model.fit(X_train_poly, Y_train)

LinearRegression()

In [11]:
# Train a model using statsmodels for p-values
model_sm = sm.OLS(Y_train, X_train_poly).fit()

In [12]:
feature_names = poly.get_feature_names_out(input_features=X.columns)
coefficients = model.coef_[0]  # Flatten the coefficients array
intercept = model.intercept_

In [13]:
coefficients = model.coef_
intercept = model.intercept_

In [14]:
Y_pred = model.predict(X_test_poly)

In [15]:
# Predict and evaluate the model
Y_pred = model.predict(X_test_poly)
mse = mean_squared_error(Y_test, Y_pred)
print(f"Mean Squared Error: {mse}")
r2 = r2_score(Y_test, Y_pred)
print(f"R² Value: {r2}")

Mean Squared Error: 11768.70951830048
R² Value: 0.9112452484805961


In [17]:
# Get feature names and coefficients
feature_names = poly.get_feature_names_out(input_features=X.columns)
feature_names = ['Intercept'] + list(feature_names)  # Add intercept to feature names
coefficients = model_sm.params  # Coefficients from statsmodels
p_values = model_sm.pvalues  # p-values from statsmodels

# Print coefficients, p-values, and significance
print("\nCoefficients, p-values, and significance:")
for name, coef, pval in zip(feature_names, coefficients, p_values):
    significance = "Significant" if pval < 0.05 else "Not Significant"
    print(f"{name}: Coefficient = {coef:.4f}, p-value = {pval:.4f} ({significance})")

# Print intercept separately
print(f"\nIntercept: {model_sm.params[0]:.4f}")


Coefficients, p-values, and significance:
Intercept: Coefficient = -4961.2470, p-value = 0.0000 (Significant)
1: Coefficient = -4491.3582, p-value = 0.0000 (Significant)
Material (M): Coefficient = 1788.5315, p-value = 0.0002 (Significant)
Electrolyte: Coefficient = 1032.2462, p-value = 0.0001 (Significant)
E_Concentration (M): Coefficient = 3854.0245, p-value = 0.0001 (Significant)
Is_Binder: Coefficient = -705.1489, p-value = 0.1254 (Not Significant)
Morphology_Encoded: Coefficient = -132.9638, p-value = 0.0110 (Significant)
Current_Density (A/g): Coefficient = -3551.5805, p-value = 0.0000 (Significant)
Material (M)^2: Coefficient = 3335.7710, p-value = 0.0000 (Significant)
Material (M) Electrolyte: Coefficient = 670.6352, p-value = 0.0000 (Significant)
Material (M) E_Concentration (M): Coefficient = 4323.9134, p-value = 0.0001 (Significant)
Material (M) Is_Binder: Coefficient = 64.1317, p-value = 0.3068 (Not Significant)
Material (M) Morphology_Encoded: Coefficient = -1.5776, p-val

<ipython-input-17-196c15e53a53>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"\nIntercept: {model_sm.params[0]:.4f}")
